In [ ]:
# ============================================================================
#  Construction du tableau QSAR : SMILES + matrice de COMPTAGE des TOP_K ECFP
#  les plus fréquents (rayon >= MIN_RADIUS uniquement) + HOMO / LUMO / GAP
#
#  POOL : on garde ici les TOP_K = 1000 ECFP les PLUS FRÉQUENTS.
#         La réduction aux 200 plus IMPACTANTS (|r| sur résidu π) se fait
#         plus bas, séparément pour chaque cible (HOMO / LUMO / GAP).
# ============================================================================

import csv
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors

RDLogger.DisableLog("rdApp.*")

# ---- Configuration ---------------------------------------------------------
CSV_PATH    = "../01_datasets/molecules_PubChemQC.csv"
RADIUS      = 2          # rayon Morgan : 2 = ECFP4, 3 = ECFP6, 4 = ECFP8
MIN_RADIUS  = 0          # on exclut les bits de rayon < 0 (atomes isolés, paires)
TOP_K       = 1000       # POOL des plus fréquents (réduit à 200/cible plus bas)
LIMIT       = 1000000
OUT_PARQUET = "pi2stage_dataset_ecfp1000.parquet"
OUT_BITS    = "pi2stage_bits_ecfp1000.csv"


def iter_rows(path, limit=None):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if limit is not None and i >= limit:
                break
            def _f(x):
                try: return float(x)
                except (TypeError, ValueError): return np.nan
            yield (row["SMILES"],
                   _f(row["DFT:HOMO_ENERGY"]),
                   _f(row["DFT:LUMO_ENERGY"]),
                   _f(row["DFT:HOMO_LUMO_GAP"]))


def ecfp4_counts(smiles):
    """Dict {bit_id: count} pour les bits de rayon >= MIN_RADIUS seulement."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    bi = {}
    fp = rdMolDescriptors.GetMorganFingerprint(mol, RADIUS, bitInfo=bi)
    counts = fp.GetNonzeroElements()
    return {bid: cnt for bid, cnt in counts.items()
            if bi[bid][0][1] >= MIN_RADIUS}

In [ ]:
# ============================================================================
#  CHARGEMENT RAPIDE — saute les deux passes de calcul ECFP
#  Lancer UNIQUEMENT cette cellule suffit pour reprendre à partir des analyses
#  (nécessite que le parquet 1000-features ait déjà été construit/sauvé)
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolDescriptors

RDLogger.DisableLog("rdApp.*")

OUT_PARQUET = "pi2stage_dataset_ecfp1000.parquet"
OUT_BITS    = "pi2stage_bits_ecfp1000.csv"

RADIUS = 2

df      = pd.read_parquet(OUT_PARQUET)
bits_df = pd.read_csv(OUT_BITS)

# Variables attendues par les cellules suivantes
bit_cols   = [c for c in df.columns if c.startswith("ECFP4_")]
top_ids    = [int(c.replace("ECFP4_", "")) for c in bit_cols]
col_index  = {bid: j for j, bid in enumerate(top_ids)}
ecfp_cols  = bit_cols
smiles_col = df["SMILES"].values

print(f"df chargé          : {df.shape[0]:,} molécules × {df.shape[1]} colonnes")
print(f"bits_df chargé     : {len(bits_df)} sous-structures (POOL)")
print(f"Colonnes ECFP4     : {len(ecfp_cols)}")
print(f"Cibles disponibles : HOMO, LUMO, GAP")

In [ ]:
# ---- PASSE 1 : fréquence documentaire de chaque sous-structure -------------
# (dans combien de molécules chaque sous-structure ECFP4 apparaît-elle)
print("Passe 1/2 — comptage des sous-structures ECFP4 ...")
doc_freq = Counter()
n_rows = 0
n_bad  = 0
for smiles, *_ in tqdm(iter_rows(CSV_PATH, LIMIT), unit=" mol"):
    n_rows += 1
    counts = ecfp4_counts(smiles)
    if counts is None:
        n_bad += 1
        continue
    # fréquence documentaire : on compte une fois par molécule, peu importe
    # le nombre d'occurrences dans la molécule
    doc_freq.update(counts.keys())

print(f"  molécules lues        : {n_rows:,}")
print(f"  SMILES non lisibles   : {n_bad:,}")
print(f"  sous-structures unique: {len(doc_freq):,}")

# ---- Sélection des TOP_K sous-structures les plus présentes ----------------
# tri : fréquence décroissante, puis id croissant (résultat déterministe)
top = sorted(doc_freq.items(), key=lambda kv: (-kv[1], kv[0]))[:TOP_K]
top_ids   = [bit_id for bit_id, _ in top]
col_index = {bit_id: j for j, bit_id in enumerate(top_ids)}
bit_cols  = [f"ECFP4_{bit_id}" for bit_id in top_ids]
print(f"  -> {len(top_ids)} sous-structures retenues "
      f"(de {top[0][1]:,} à {top[-1][1]:,} molécules)")

# table de référence des bits retenus
bits_df = pd.DataFrame({
    "rank":      np.arange(1, len(top_ids) + 1),
    "ecfp4_id":  top_ids,
    "doc_count": [c for _, c in top],
    "fraction":  [c / n_rows for _, c in top],
})

# ---- PASSE 2 : matrice de COMPTAGE + métadonnées --------------------------
# uint16 : autorise jusqu'à 65 535 occurrences d'une même sous-structure
# dans une seule molécule (largement suffisant)
print("Passe 2/2 — construction de la matrice de comptage ...")
counts_matrix = np.zeros((n_rows, len(top_ids)), dtype=np.uint16)
smiles_col = np.empty(n_rows, dtype=object)
homo = np.empty(n_rows, dtype=np.float64)
lumo = np.empty(n_rows, dtype=np.float64)
gap  = np.empty(n_rows, dtype=np.float64)

for i, (smiles, h, l, g) in enumerate(
        tqdm(iter_rows(CSV_PATH, LIMIT), total=n_rows, unit=" mol")):
    smiles_col[i] = smiles
    homo[i], lumo[i], gap[i] = h, l, g
    counts = ecfp4_counts(smiles)
    if counts is None:
        continue
    for bit_id, n in counts.items():
        j = col_index.get(bit_id)
        if j is not None:
            counts_matrix[i, j] = n         # ← VRAI COMPTE, pas 0/1

# ---- Assemblage du tableau final ------------------------------------------
df = pd.DataFrame(counts_matrix, columns=bit_cols, copy=False)
df.insert(0, "SMILES", smiles_col)
df["HOMO"] = homo
df["LUMO"] = lumo
df["GAP"]  = gap

print(f"\nTableau final : {df.shape[0]:,} lignes x {df.shape[1]:,} colonnes")
print(f"Valeur max dans la matrice de comptage : {counts_matrix.max()}")
print(f"Nb de cellules > 1 : {(counts_matrix > 1).sum():,} "
      f"({(counts_matrix > 1).sum() / counts_matrix.size * 100:.2f} %)")
df.head()


In [ ]:
# ---- Sauvegarde ------------------------------------------------------------
# Parquet : compact et rapide à relire (df.read_parquet). Le CSV serait
# énorme (~1000 colonnes x ~880k lignes) et lent ; à éviter ici.
df.to_parquet(OUT_PARQUET, index=False)
bits_df.to_csv(OUT_BITS, index=False)

print(f"Tableau    -> {OUT_PARQUET}")
print(f"Bits ECFP4 -> {OUT_BITS}")
print("\nAperçu des sous-structures les plus présentes :")
bits_df.head(10)


In [ ]:
# ============================================================================
#  STAGE 1 — largest π-chain size  (PI_CHAIN_SIZE)  +  residualization
#
#  "Plus grande chaîne π" — same definition as the report (gap_pi_relation):
#     an atom is π-active if it is AROMATIC or hybridised SP2/SP ;
#     PI_CHAIN_SIZE = number of atoms in the LARGEST connected π component.
#     (this is the π-SYSTEM SIZE, NOT the π-electron count used before.)
#
#  We regress HOMO / LUMO / GAP on log(PI_CHAIN_SIZE) and keep the RESIDUALS.
#  Every ECFP analysis below then runs on these residuals, so the "most
#  impacting ECFP" reflect LOCAL effects orthogonal to the global π trend.
#
#  Molecules without a π system (PI_CHAIN_SIZE = 0) : log feature = 0, so the
#  stage-1 prediction is just the intercept and the residual ≈ raw value.
#
#  ⚠ Run this cell AFTER df exists (fast-load cell OR the build passes above).
# ============================================================================
import os
from tqdm import tqdm
from rdkit.Chem import rdchem
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes as _inset_axes

PI_CACHE = "pi2stage_pi_chain_size.npy"

TARGETS       = ["HOMO", "LUMO", "GAP"]
TARGET_COLORS = {"HOMO": "#F472B6", "LUMO": "#7B52AB", "GAP": "#4472C4"}


def largest_pi_chain(smiles):
    """Size (atom count) of the largest connected π component.
    π-active atom = aromatic OR hybridised SP2/SP (report definition)."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0
    SP2, SP = rdchem.HybridizationType.SP2, rdchem.HybridizationType.SP
    pi_set = {a.GetIdx() for a in mol.GetAtoms()
              if a.GetIsAromatic() or a.GetHybridization() in (SP2, SP)}
    if not pi_set:
        return 0
    visited, best = set(), 0
    for start in pi_set:
        if start in visited:
            continue
        size, stack = 0, [start]
        visited.add(start)
        while stack:
            u = stack.pop(); size += 1
            for nb in mol.GetAtomWithIdx(u).GetNeighbors():
                v = nb.GetIdx()
                if v in pi_set and v not in visited:
                    visited.add(v); stack.append(v)
        best = max(best, size)
    return best


# ---- Compute PI_CHAIN_SIZE (cached on disk, aligned to df row order) -------
pi_size = None
if os.path.exists(PI_CACHE):
    cached = np.load(PI_CACHE)
    if len(cached) == len(df):
        pi_size = cached
        print(f"PI_CHAIN_SIZE loaded from cache : {PI_CACHE}")

if pi_size is None:
    print("Computing PI_CHAIN_SIZE for every molecule (cached afterwards) …")
    pi_size = np.fromiter(
        (largest_pi_chain(s) for s in tqdm(df["SMILES"].values, unit=" mol")),
        dtype=np.int32, count=len(df),
    )
    np.save(PI_CACHE, pi_size)
    print(f"Saved -> {PI_CACHE}")

df["PI_CHAIN_SIZE"] = pi_size
df["LOG_PI"]        = np.log(np.clip(pi_size, 1, None))   # 0 when no π system

print(f"\nPI_CHAIN_SIZE : min={pi_size.min()}  median={int(np.median(pi_size))}  max={pi_size.max()}")
print(f"Molecules without π system (size 0) : {(pi_size == 0).sum():,}  "
      f"({(pi_size == 0).mean()*100:.1f} %)")

# ---- Stage 1 : log regression of each target on PI_CHAIN_SIZE -------------
log_pi_all = df["LOG_PI"].values.reshape(-1, 1)
fit_mask   = (
    df[TARGETS].notna().all(axis=1) &
    (df["HOMO"] >= -15) & (df["LUMO"] >= -15)
).values

pi_fit = {}
print()
for t in TARGETS:
    y = df[t].values
    m = fit_mask & np.isfinite(y)
    reg = LinearRegression().fit(log_pi_all[m], y[m])
    pred = reg.predict(log_pi_all)                 # prediction for every row
    df[f"{t}_pi_pred"] = pred
    df[f"{t}_res"]     = y - pred                   # residual = raw − π trend
    r2 = r2_score(y[m], pred[m])
    pi_fit[t] = dict(a=reg.coef_[0], b=reg.intercept_, r2=r2)
    print(f"  {t}: y = {reg.coef_[0]:+.4f}·ln(PI_CHAIN_SIZE) {reg.intercept_:+.4f}"
          f"   R² = {r2:.4f}")

# ---- Plot : HOMO/LUMO/GAP vs PI_CHAIN_SIZE (same style as n(e-π) figures) --
_cmap = LinearSegmentedColormap.from_list(
    "candy_pop", ["#e9f6ff", "#f0d7ff", "#cd9cf7b3", "#f465bf", "#c71585"])

def _dens(x, y, bins=60):
    H, xe, ye = np.histogram2d(x, y, bins=bins)
    xi = np.clip(np.digitize(x, xe) - 1, 0, bins - 1)
    yi = np.clip(np.digitize(y, ye) - 1, 0, bins - 1)
    return H[xi, yi]

def _marg(ax_ref, ax_ins, values, orientation, n_bins=30):
    cnt, edg = np.histogram(values, bins=n_bins)
    ctr = 0.5 * (edg[:-1] + edg[1:]); w = edg[1:] - edg[:-1]
    if orientation == "y":
        ax_ins.barh(ctr, cnt, height=w*0.9, color="#e8cdffb3", lw=0, alpha=0.9)
        ax_ins.invert_xaxis(); ax_ins.set_ylim(ax_ref.get_ylim())
    else:
        ax_ins.bar(ctr, cnt, width=w*0.9, color="#e8cdffb3", lw=0, alpha=0.9)
        ax_ins.set_xlim(ax_ref.get_xlim())
    ax_ins.set_xticks([]); ax_ins.set_yticks([]); ax_ins.patch.set_alpha(0.0)
    for sp in ax_ins.spines.values():
        sp.set_visible(False)

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), facecolor="white")
fig.suptitle("Stage 1 — HOMO / LUMO / GAP vs largest π-chain size  (logarithmic fit)",
             fontsize=13, fontweight="bold", y=1.04)

# Filter DFT outliers
outlier_mask = (df["HOMO"] >= -15) & (df["LUMO"] >= -15)
n_removed = (~outlier_mask).sum()
print(f"\nDFT outliers removed: {n_removed:,}")

for ax, t in zip(axes, TARGETS):
    m = fit_mask & np.isfinite(df[t].values) & (pi_size > 0) & outlier_mask
    x = pi_size[m].astype(float)
    y = df[t].values[m]
    if len(x) > 200_000:
        idx = rng.choice(len(x), 200_000, replace=False)
        xp, yp = x[idx], y[idx]
    else:
        xp, yp = x, y
    z = _dens(xp, yp, bins=60); o = np.argsort(z)
    ax.scatter(xp[o], yp[o], c=z[o], cmap=_cmap, s=6, alpha=0.7, rasterized=True)

    a, b, r2 = pi_fit[t]["a"], pi_fit[t]["b"], pi_fit[t]["r2"]
    xs = np.linspace(x.min(), x.max(), 200)
    ax.plot(xs, a*np.log(xs) + b, color="black", lw=2, zorder=5)

    ax.set_xlabel("largest π-chain size  (atoms)", fontsize=10)
    ax.set_ylabel(f"{t} (eV)", fontsize=10, color=TARGET_COLORS[t], fontweight="bold")
    ax.set_title(t, fontsize=12, fontweight="bold", color=TARGET_COLORS[t])
    ax.text(0.04, 0.96,
            f"R² = {r2:.4f}\nn = {int(m.sum()):,}\ny = {a:+.3f}·ln(x) {b:+.3f}",
            transform=ax.transAxes, fontsize=9, va="top", family="monospace",
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=4))
    ax.set_facecolor("white")

    ax_y = _inset_axes(ax, width="15%", height="100%", loc="center right",
                       bbox_to_anchor=(0, 0, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
    _marg(ax, ax_y, yp, "y")
    ax_x = _inset_axes(ax, width="100%", height="15%", loc="lower center",
                       bbox_to_anchor=(0, 0, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
    _marg(ax, ax_x, xp, "x")

plt.tight_layout()
plt.savefig("pi2stage_pi_chain_regression.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print("Figure -> pi2stage_pi_chain_regression.png")

In [ ]:
# ============================================================================
#  Influence of the 1000 ECFP4 pool on the π-RESIDUALS of HOMO, LUMO, GAP
#  + SELECTION of the 200 most impacting ECFP PER TARGET (by |r| on residual)
#
#  Pool   : 1000 most frequent ECFP4 (built above).
#  Select : for each target, keep the 200 with the strongest |r| on residual
#           → selected_bits["HOMO" / "LUMO" / "GAP"]  (3 distinct feature sets).
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap

TARGET_COLORS = {"HOMO": "#F472B6", "LUMO": "#7B52AB", "GAP": "#4472C4"}
TARGET_CMAPS  = {
    t: LinearSegmentedColormap.from_list(f"cm_{t}", ["#ffffff", TARGET_COLORS[t]])
    for t in TARGET_COLORS
}

SELECT_K = 200   # nb d'ECFP retenus par cible (parmi le pool de 1000)

ecfp_cols = [c for c in df.columns if c.startswith("ECFP4_")]
print(f"POOL ECFP4 disponible : {len(ecfp_cols)} (les plus fréquents)\n")

# on a besoin des cibles ET de leurs résidus de la chaîne π
res_cols = [f"{t}_res" for t in ("HOMO", "LUMO", "GAP")]
mask = (
    df[["HOMO", "LUMO", "GAP"] + res_cols].notna().all(axis=1) &
    (df["HOMO"] >= -15) & (df["LUMO"] >= -15)
)
df_c = df[mask].reset_index(drop=True)
print(f"Molecules used: {len(df_c):,}  ({mask.mean()*100:.1f} %)\n")

TOP = 20

corr        = {}   # |r| trié desc sur tout le pool — pour graphiques / sélection
corr_signed = {}   # r signé        — pour connaître le sens de l'effet
# NB : corrélation avec les RÉSIDUS {target}_res (et non la valeur brute) pour
#      isoler les effets ECFP locaux orthogonaux à la taille du système π.
for target in ("HOMO", "LUMO", "GAP"):
    raw = df_c[ecfp_cols].corrwith(df_c[f"{target}_res"])
    corr_signed[target] = raw
    corr[target]        = raw.abs().sort_values(ascending=False)

# ---- Sélection des 200 plus impactants PAR CIBLE --------------------------
selected_bits = {t: corr[t].head(SELECT_K).index.tolist() for t in ("HOMO", "LUMO", "GAP")}
print(f"Sélection des {SELECT_K} ECFP les plus impactants par cible (|r| sur résidu) :")
for t in ("HOMO", "LUMO", "GAP"):
    seuil = corr[t].iloc[SELECT_K-1]
    inter = len(set(selected_bits[t]) & set(selected_bits["GAP"]))
    print(f"  {t:<4}: {len(selected_bits[t])} features  (|r| min retenu = {seuil:.4f})")
n_union = len(set().union(*selected_bits.values()))
print(f"  → union des 3 jeux : {n_union} ECFP distincts\n")

for target, series in corr.items():
    top = series.head(TOP).reset_index()
    top.columns = ["ECFP4", f"|r| with {target} residual"]
    top.index = range(1, TOP + 1)
    print(f"{'━'*54}")
    print(f"  Top {TOP} ECFP4  —  influence on {target} residual (π removed)")
    print(f"{'━'*54}")
    try:
        display(top)
    except NameError:
        print(top.to_string())
    print()

fig, axes = plt.subplots(1, 3, figsize=(20, 9), facecolor="white")
fig.suptitle("Influence of ECFP4 substructures on π-residual HOMO, LUMO and GAP\n"
             f"(pool of {len(ecfp_cols)} → top {SELECT_K} retained per target)",
             fontsize=14, fontweight="bold", y=1.02)

for ax, (target, series) in zip(axes, corr.items()):
    top    = series.head(TOP)
    labels = [c.replace("ECFP4_", "") for c in top.index]
    ypos   = range(TOP, 0, -1)
    colors = TARGET_CMAPS[target](np.linspace(0.35, 1.0, TOP))

    bars = ax.barh(list(ypos), top.values, color=colors, alpha=0.90)

    ax.set_yticks(list(ypos))
    ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_xlabel("|Pearson correlation| with residual", fontsize=10)
    ax.set_title(f"Top {TOP} ECFP4  →  {target} residual",
                 fontsize=13, fontweight="bold", color=TARGET_COLORS[target])
    ax.set_xlim(0, top.values[0] * 1.15)
    ax.set_facecolor("white")

    for bar, val in zip(bars, top.values):
        ax.text(val + top.values[0] * 0.015,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=6.5)

plt.tight_layout()
plt.savefig("pi2stage_ecfp_influence.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.show()
print("Figure -> pi2stage_ecfp_influence.png")

In [ ]:
# ============================================================================
#  Concrete effect of ECFP4 bits on the π-RESIDUAL HOMO / LUMO / GAP
#  Top 6 bits per target (by |r| on residual, within the 200 selected)
#  + ligne pointillée = moyenne globale du résidu
# ============================================================================
import matplotlib.colors as mcolors

TARGET_COLORS = {"HOMO": "#F472B6", "LUMO": "#7B52AB", "GAP": "#4472C4"}

N_BOX = 6

df_box = df_c[(df_c["HOMO"] >= -15) & (df_c["LUMO"] >= -15)].reset_index(drop=True)
print(f"Molecules used for box plots: {len(df_box):,}  "
      f"({len(df_c)-len(df_box)} DFT outliers removed)")

fig, axes = plt.subplots(3, N_BOX, figsize=(18, 10), facecolor="white")
fig.suptitle("Distribution of π-residual HOMO / LUMO / GAP when ECFP4 bit is present\n"
             "(dashed line = global residual mean)",
             fontsize=13, fontweight="bold", y=1.02)

for row_i, (target, ser) in enumerate(corr.items()):
    res_col     = f"{target}_res"
    top_bits    = ser.head(N_BOX).index.tolist()
    base_col    = TARGET_COLORS[target]
    global_mean = df_box[res_col].mean()

    for col_i, feat in enumerate(top_bits):
        ax = axes[row_i, col_i]

        present = df_box.loc[df_box[feat] > 0, res_col].dropna()

        bp = ax.boxplot(
            [present],
            tick_labels=["present"],
            patch_artist=True,
            widths=0.5,
            medianprops=dict(color="white", linewidth=2.0),
            whiskerprops=dict(color="#4a4a4a", lw=1.2),
            capprops=dict(color="#4a4a4a", lw=1.2),
            flierprops=dict(marker="o", color=base_col, markersize=2.5,
                            alpha=0.4, linestyle="none"),
        )
        bp["boxes"][0].set_facecolor(mcolors.to_rgba(base_col, alpha=0.75))
        bp["boxes"][0].set_edgecolor("#4a4a4a")
        bp["boxes"][0].set_linewidth(1.0)

        ax.axhline(global_mean, linestyle="--", color="#555", linewidth=1.2,
                   zorder=5)

        ax.set_facecolor("white")
        delta = present.median() - global_mean
        ax.set_title(f"{feat.replace('ECFP4_', '')}\nΔmed={delta:+.2f} eV",
                     fontsize=7.5)
        ax.tick_params(axis="x", labelsize=7)
        if col_i == 0:
            ax.set_ylabel(f"{target} residual", fontsize=10, fontweight="bold",
                          color=TARGET_COLORS[target])
        ax.yaxis.set_tick_params(labelsize=7)

plt.tight_layout()
plt.savefig("pi2stage_ecfp_boxplots.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.show()
print("Figure -> pi2stage_ecfp_boxplots.png")

In [ ]:
# ============================================================================
#  TWO-STAGE regression  HOMO / LUMO / GAP  —  200 features PER TARGET
#    Stage 1 : y = a·ln(PI_CHAIN_SIZE) + b           (fit above, pi_fit)
#    Stage 2 : Ridge on the 200 ECFP4 most impacting for THIS target
#              (selected_bits[target]) trained on the π-residual y_res
#    Final   : ŷ = stage1 + Ridge(ECFP4)
#  Each target uses its own feature matrix; the train/test split is shared.
# ============================================================================
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes as _inset_axes

N_FORMULA    = 15
MIN_ECFP_BITS = 3   # molécules avec moins de MIN_ECFP_BITS bits présents (pool) exclues

TARGET_COLORS = {"HOMO": "#F472B6", "LUMO": "#7B52AB", "GAP": "#4472C4"}

cmap_density = LinearSegmentedColormap.from_list(
    "candy_pop", ["#e9f6ff", "#f0d7ff", "#cd9cf7b3", "#f465bf", "#c71585"])

def local_density(x, y, bins=60):
    H, x_e, y_e = np.histogram2d(x, y, bins=bins)
    xi = np.clip(np.digitize(x, x_e) - 1, 0, bins - 1)
    yi = np.clip(np.digitize(y, y_e) - 1, 0, bins - 1)
    return H[xi, yi]

def _marginal_hist(ax_ref, ax_inset, values, orientation, n_bins=30):
    counts, edges = np.histogram(values, bins=n_bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths  = edges[1:] - edges[:-1]
    if orientation == "y":
        ax_inset.barh(centers, counts, height=widths * 0.9,
                      color="#e8cdffb3", linewidth=0, alpha=0.9)
        ax_inset.invert_xaxis()
        ax_inset.set_ylim(ax_ref.get_ylim())
    else:
        ax_inset.bar(centers, counts, width=widths * 0.9,
                     color="#e8cdffb3", linewidth=0, alpha=0.9)
        ax_inset.set_xlim(ax_ref.get_xlim())
    ax_inset.set_xticks([]); ax_inset.set_yticks([])
    for sp in ax_inset.spines.values():
        sp.set_visible(False)
    ax_inset.patch.set_alpha(0.0)

# ---- Filtres : DFT outliers + au moins MIN_ECFP_BITS bits présents (pool) ---
before = len(df_c)
df_reg = df_c[(df_c["HOMO"] >= -15) & (df_c["LUMO"] >= -15)].reset_index(drop=True)
n_bits_present = (df_reg[ecfp_cols] > 0).sum(axis=1)
df_reg = df_reg[n_bits_present >= MIN_ECFP_BITS].reset_index(drop=True)

print(f"Molecules avant filtre       : {before:,}")
print(f"Après filtre DFT outliers    : {(df_c['HOMO'] >= -15).sum():,}")
print(f"Après filtre ECFP ≥ {MIN_ECFP_BITS} bits  : {len(df_reg):,}  "
      f"({before - len(df_reg)} retirées au total)\n")

TARGETS = ["HOMO", "LUMO", "GAP"]

# split partagé (indices) — identique à la cellule des outliers
idx_tr, idx_te = train_test_split(
    np.arange(len(df_reg)), test_size=0.2, random_state=42)

# ---- Training : une régression 2-stage par cible, features dédiées ---------
results = {}
for target in TARGETS:
    cols    = selected_bits[target]                      # 200 features de la cible
    Xt      = df_reg[cols].values.astype(np.float32)
    y_true  = df_reg[target].values
    y_res   = df_reg[f"{target}_res"].values             # cible du stage 2
    pi_pred = df_reg[f"{target}_pi_pred"].values          # prédiction stage 1

    X_tr, X_te     = Xt[idx_tr], Xt[idx_te]
    yres_tr        = y_res[idx_tr]
    pi_tr, pi_te   = pi_pred[idx_tr], pi_pred[idx_te]
    ytrue_tr, ytrue_te = y_true[idx_tr], y_true[idx_te]

    model = Ridge(alpha=1.0)
    model.fit(X_tr, yres_tr)                              # stage 2 : ECFP → résidu
    res_pred_tr = model.predict(X_tr)
    res_pred_te = model.predict(X_te)

    # prédiction combinée = stage 1 + stage 2
    y_pred_tr = pi_tr + res_pred_tr
    y_pred_te = pi_te + res_pred_te

    results[target] = dict(
        model=model, cols=cols,
        y_tr=ytrue_tr, y_te=ytrue_te,
        y_pred_tr=y_pred_tr, y_pred_te=y_pred_te,
        r2_tr=r2_score(ytrue_tr, y_pred_tr),
        r2_te=r2_score(ytrue_te, y_pred_te),
        rmse_tr=np.sqrt(mean_squared_error(ytrue_tr, y_pred_tr)),
        rmse_te=np.sqrt(mean_squared_error(ytrue_te, y_pred_te)),
        mae_tr=mean_absolute_error(ytrue_tr, y_pred_tr),
        mae_te=mean_absolute_error(ytrue_te, y_pred_te),
        res_r2_te=r2_score(y_res[idx_te], res_pred_te),
    )

# ---- Equations : stage 1 (π) + top stage 2 (ECFP sur résidus) -------------
for target in TARGETS:
    r     = results[target]
    m     = r["model"]
    cols  = r["cols"]
    a, b  = pi_fit[target]["a"], pi_fit[target]["b"]
    order = np.argsort(np.abs(m.coef_))[::-1]
    print(f"{'='*64}")
    print(f"  Two-stage equation — {target}   ({len(cols)} ECFP features)")
    print(f"{'='*64}")
    print(f"  {target}_predicted =")
    print(f"     stage 1 (π) : {a:+.4f}·ln(PI_CHAIN_SIZE) {b:+.4f}")
    print(f"     stage 2 (ECFP on residual, intercept {m.intercept_:+.4f}) :")
    for j in order[:N_FORMULA]:
        print(f"        {m.coef_[j]:+.4f}  x  {cols[j]}")
    print(f"        ... + {len(cols)-N_FORMULA} other terms")
    print(f"  R2   train={r['r2_tr']:.4f}  test={r['r2_te']:.4f}  |  "
          f"RMSE train={r['rmse_tr']:.4f}  test={r['rmse_te']:.4f} eV  |  "
          f"MAE  train={r['mae_tr']:.4f}  test={r['mae_te']:.4f} eV")
    print(f"  (stage 2 alone — R² on residual, test = {r['res_r2_te']:.4f})\n")

metrics_df = pd.DataFrame({
    "Target":      TARGETS,
    "R2 stage1 π": [round(pi_fit[t]["r2"],       4) for t in TARGETS],
    "R2 test":     [round(results[t]["r2_te"],   4) for t in TARGETS],
    "RMSE test":   [round(results[t]["rmse_te"], 4) for t in TARGETS],
    "MAE test":    [round(results[t]["mae_te"],  4) for t in TARGETS],
    "R2 ECFP|res": [round(results[t]["res_r2_te"],4) for t in TARGETS],
}).set_index("Target")
try:
    display(metrics_df)
except NameError:
    print(metrics_df.to_string())

# ---- 3 x 2 scatter grid with density coloring (combined prediction) --------
rng   = np.random.default_rng(0)
N_MAX = 10000000

fig, axes = plt.subplots(3, 2, figsize=(14, 18), facecolor="white")
fig.suptitle("Predicted vs Actual — two-stage  π-chain + Ridge ECFP4 (200 feat./target)\n"
             "(DFT outliers + ECFP < 3 bits removed)",
             fontsize=14, fontweight="bold", y=1.01)

col_titles = ["Train (80 %)", "Test (20 %)"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight="bold", pad=12)

for row_i, target in enumerate(TARGETS):
    r = results[target]
    y_tr = r["y_tr"];  y_prd_tr = r["y_pred_tr"]
    y_te = r["y_te"];  y_prd_te = r["y_pred_te"]

    idx_tr_p = rng.choice(len(y_tr), size=min(N_MAX, len(y_tr)), replace=False)
    idx_te_p = rng.choice(len(y_te), size=min(N_MAX, len(y_te)), replace=False)

    all_v  = np.concatenate([y_tr, y_prd_tr, y_te, y_prd_te])
    lo, hi = all_v.min() - 0.3, all_v.max() + 0.3
    diag   = [lo, hi]

    for col_j, (y_real, y_pred, r2_val, rmse_val, mae_val) in enumerate([
        (y_tr[idx_tr_p], y_prd_tr[idx_tr_p], r["r2_tr"], r["rmse_tr"], r["mae_tr"]),
        (y_te[idx_te_p], y_prd_te[idx_te_p], r["r2_te"], r["rmse_te"], r["mae_te"]),
    ]):
        ax = axes[row_i, col_j]

        z     = local_density(y_real, y_pred, bins=40)
        order = np.argsort(z)
        sc    = ax.scatter(y_real[order], y_pred[order], c=z[order],
                           cmap=cmap_density, s=8, alpha=0.75, rasterized=True)
        plt.colorbar(sc, ax=ax, label="Density", pad=0.02, fraction=0.030)

        ax.plot(diag, diag, color=TARGET_COLORS[target], lw=1.8, zorder=5,
                label="predicted = actual")
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
        ax.set_facecolor("white")

        ax.set_ylabel(f"{target} predicted (eV)", fontsize=10,
                      color=TARGET_COLORS[target], fontweight="bold")
        ax.set_xlabel(f"{target} actual (eV)", fontsize=9)

        note = (f"R2   = {r2_val:.4f}\n"
                f"RMSE = {rmse_val:.4f} eV\n"
                f"MAE  = {mae_val:.4f} eV\n"
                f"n    = {len(y_real):,}")
        ax.text(0.04, 0.97, note, transform=ax.transAxes,
                fontsize=8, va="top", family="monospace",
                bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=4))

        ax_y = _inset_axes(ax, width="18%", height="100%", loc="center right",
                           bbox_to_anchor=(0.0, 0, 1, 1),
                           bbox_transform=ax.transAxes, borderpad=0)
        _marginal_hist(ax, ax_y, y_pred, "y")

        ax_x = _inset_axes(ax, width="100%", height="18%", loc="lower center",
                           bbox_to_anchor=(0, 0, 1, 1),
                           bbox_transform=ax.transAxes, borderpad=0)
        _marginal_hist(ax, ax_x, y_real, "x")

plt.tight_layout()
plt.savefig("pi2stage_twostage_prediction.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()
print("Figure -> pi2stage_twostage_prediction.png")

In [ ]:
import io
from tqdm.notebook import tqdm
from rdkit.Chem import Draw, AllChem
from IPython.display import display as ipy_display, Image as IPyImage
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage

# NB : `corr` / `corr_signed` portent sur les RÉSIDUS de la chaîne π, donc ces
#      fragments sont les "most impacting ECFP" (top 30/cible) parmi les 200 retenus.

N_TOP        = 30
MOLS_PER_ROW = 5

# ---- Collecte des colonnes à illustrer -------------------------------------
bits_needed_cols = set()
for ser in corr.values():
    for bit_col in ser.head(N_TOP).index:
        bits_needed_cols.add(bit_col)
# Inclure aussi tous les selected_bits (utilisés dans la cellule boxplots)
for bits in selected_bits.values():
    bits_needed_cols.update(bits)
print(f"Bits à illustrer : {len(bits_needed_cols)} (top-{N_TOP}/cible + selected_bits)")

# ---- Recherche ciblée : pour chaque bit, on cherche directement dans df_c --
# df_c[col] > 0  → liste des molécules qui ont ce bit → on prend les 50 premiers
# et on vérifie avec GetMorganFingerprint (évite les problèmes de scan linéaire)
example_mol = {}
missing     = []

for bit_col in tqdm(sorted(bits_needed_cols), desc="Recherche d'exemples"):
    bid = int(bit_col.replace("ECFP4_", ""))
    if bit_col not in df_c.columns:
        missing.append(bit_col); continue

    candidates = df_c.index[df_c[bit_col] > 0].tolist()
    found = False
    for idx in candidates[:100]:
        mol = Chem.MolFromSmiles(df_c.at[idx, "SMILES"])
        if mol is None:
            continue
        bi = {}
        rdMolDescriptors.GetMorganFingerprint(mol, RADIUS, bitInfo=bi)
        if bid in bi:
            example_mol[bid] = (mol, bi)
            found = True
            break
    if not found:
        missing.append(bit_col)

print(f"✓ {len(example_mol)}/{len(bits_needed_cols)} bits illustrés")
if missing:
    print(f"  Non trouvés ({len(missing)}) : {missing[:5]}{'...' if len(missing)>5 else ''}")
print()

# ---- Figures par cible (DrawMorganEnvs — notation blog RDKit) ---------------
SUB_W, SUB_H = 360, 300

for target, ser in corr.items():
    print(f"\n{'━'*80}\n  Top {N_TOP} fragments  →  {target} residual (π removed)\n{'━'*80}")
    rows, envs, legends = [], [], []

    for rank, (bit_col, abs_r) in enumerate(ser.head(N_TOP).items(), start=1):
        bid       = int(bit_col.replace("ECFP4_", ""))
        signed_r  = corr_signed[target].loc[bit_col]
        direction = "↑ +" if signed_r > 0 else "↓ −"
        n_occ     = int((df[bit_col] > 0).sum())

        rows.append({"Rang": rank, "r signé": round(signed_r, 4),
                     "effet": direction, "n mol": n_occ})

        if bid in example_mol:
            mol_ex, bi_ex = example_mol[bid]
            aix_ex, r_ex  = bi_ex[bid][0]
            sign_label    = f"+{signed_r:.3f}" if signed_r > 0 else f"{signed_r:.3f}"
            envs.append((mol_ex, aix_ex, r_ex))
            legends.append(f"#{rank}  r={sign_label}  n={n_occ:,}")

    sub = pd.DataFrame(rows); sub.index = range(1, len(sub)+1)
    try:    display(sub)
    except: print(sub.to_string())

    if envs:
        fname    = f"pi2stage_fragments_{target}.png"
        raw      = Draw.DrawMorganEnvs(
            envs, molsPerRow=MOLS_PER_ROW,
            subImgSize=(SUB_W, SUB_H), legends=legends,
            useSVG=False,
        )
        png_bytes = raw if isinstance(raw, bytes) else raw.data
        pil_img   = PILImage.open(io.BytesIO(png_bytes))

        fig_w = pil_img.width  / 120
        fig_h = pil_img.height / 120 + 0.7
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), facecolor="white")
        ax.imshow(np.array(pil_img))
        ax.axis("off")
        ax.set_title(
            f"{target} residual", fontsize=18, fontweight="bold",
            color=TARGET_COLORS[target], fontfamily="Arial", pad=10
        )
        plt.tight_layout()
        plt.savefig(fname, dpi=300, bbox_inches="tight", facecolor="white")
        plt.show()
        print(f"Figure → {fname}")


**Figure — Top ECFP4 environments ranked by absolute Pearson correlation $|r|$ with HOMO, LUMO and GAP, computed on 882 924 molecules from PubChemQC.**  
Visualization inspired by the [RDKit blog post on chemical words](https://greglandrum.github.io/rdkit-blog/posts/2023-01-08-chemical-words.html) and produced with `Draw.DrawMorganEnvs`.

Each panel shows the **source molecule** in which the circular environment was first encountered, with the ECFP4 substructure highlighted directly inside that molecule context rather than as an isolated fragment.

**Color and line encoding (RDKit defaults):**

| Element | Color | Meaning |
|---|---|---|
| **Central atom** | Light blue (RGB 0.6, 0.8, 1.0) | Atom at the center of the Morgan fingerprint bit |
| **Aromatic bonds / atoms** | Pale yellow (RGB 0.9, 0.9, 0.2) | Aromatic system within the captured environment |
| **Ring atoms** | Light gray (RGB 0.8, 0.8, 0.8) | Non-aromatic ring membership inside the environment |
| **Outer environment atoms** | Very light gray (RGB 0.9, 0.9, 0.9) | Atoms at the periphery of the circular environment |
| **Non-highlighted atoms** | White / standard RDKit palette | Atoms outside the ECFP4 environment (context only) |
| **Bonds inside environment** | Thickened, colored stroke | Bonds within the highlighted substructure |
| **Bonds outside environment** | Thin gray stroke | Context bonds not part of the fingerprint bit |

The **legend** below each structure reports: rank `#`, signed Pearson correlation `r` (+ raises / − lowers the target energy), and `n` = number of molecules in the dataset containing that environment.

In [ ]:
# ============================================================================
#  Boxplots π-residual HOMO / LUMO / GAP + structure ECFP+contexte
#  Top fragments par |Δmédiane| (sur résidu π) parmi les 200 sélectionnés/cible
# ============================================================================
import io
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import numpy as np
from PIL import Image as PILImage
from rdkit.Chem import Draw

N_BOX    = 5
MIN_FREQ = 100

df_box = df_c[(df_c["HOMO"] >= -15) & (df_c["LUMO"] >= -15)].reset_index(drop=True)
print(f"Molécules utilisées : {len(df_box):,}")

# --- Calcul du Δmédiane sur le RÉSIDU, parmi les 200 features de la cible ---
print("Calcul des effets (Δmédiane sur résidu) ...")
delta_median = {}
for target in ["HOMO", "LUMO", "GAP"]:
    res_col    = f"{target}_res"
    global_med = df_box[res_col].median()
    deltas = {}
    for feat in selected_bits[target]:
        mask = df_box[feat] > 0
        if mask.sum() < MIN_FREQ:
            continue
        deltas[feat] = df_box.loc[mask, res_col].median() - global_med
    delta_median[target] = pd.Series(deltas)

# --- Figure -----------------------------------------------------------------
fig = plt.figure(figsize=(N_BOX * 3.4, 3 * 5.5), facecolor="white")
fig.suptitle(
    "Distribution π-residual HOMO / LUMO / GAP — top fragments par |Δmédiane|\n"
    "(ligne pointillée = médiane globale du résidu)",
    fontsize=13, fontweight="bold", y=1.01,
)

outer = gridspec.GridSpec(3, 1, figure=fig, hspace=0.6)

for row_i, (target, _) in enumerate(corr.items()):
    res_col    = f"{target}_res"
    top_bits   = delta_median[target].abs().sort_values(ascending=False).head(N_BOX).index.tolist()
    base_col   = TARGET_COLORS[target]
    global_med = df_box[res_col].median()

    inner = gridspec.GridSpecFromSubplotSpec(
        2, N_BOX, subplot_spec=outer[row_i],
        height_ratios=[2.5, 1], hspace=0.05, wspace=0.35
    )

    # ---- Label HOMO / LUMO / GAP centré sur la ligne ----------------------
    label_ax = fig.add_subplot(inner[0, :])
    label_ax.set_facecolor("none")
    for sp in label_ax.spines.values():
        sp.set_visible(False)
    label_ax.set_xticks([])
    label_ax.set_yticks([])
    label_ax.set_title(
        f"{target} residual", fontsize=14, fontweight="bold",
        color=base_col, pad=6, loc="center",
        fontfamily="Arial"
    )

    for col_i, feat in enumerate(top_bits):
        bid       = int(feat.replace("ECFP4_", ""))
        present   = df_box.loc[df_box[feat] > 0, res_col].dropna()
        n_present = len(present)
        delta     = delta_median[target][feat]
        r_val     = corr_signed[target].get(feat, float("nan"))

        # ---- Boxplot -------------------------------------------------------
        ax_box = fig.add_subplot(inner[0, col_i])
        bp = ax_box.boxplot(
            [present], tick_labels=[""], patch_artist=True, widths=0.5,
            medianprops=dict(color="white", linewidth=2.0),
            whiskerprops=dict(color="#4a4a4a", lw=1.2),
            capprops=dict(color="#4a4a4a", lw=1.2),
            flierprops=dict(marker="o", color=base_col, markersize=2.0,
                            alpha=0.35, linestyle="none"),
        )
        bp["boxes"][0].set_facecolor(mcolors.to_rgba(base_col, alpha=0.75))
        bp["boxes"][0].set_edgecolor("#4a4a4a")
        ax_box.axhline(global_med, linestyle="--", color="#555", linewidth=1.2, zorder=5)
        ax_box.set_facecolor("white")
        ax_box.yaxis.set_tick_params(labelsize=7)
        if col_i == 0:
            ax_box.set_ylabel("Residual energy (eV)", fontsize=9, color="#4a4a4a")

        # ---- Image (DrawMorganEnvs — même notation que la cellule dessus) --
        ax_img = fig.add_subplot(inner[1, col_i])
        if bid in example_mol:
            mol_ex, bi_ex = example_mol[bid]
            aix_ex, r_ex  = bi_ex[bid][0]
            try:
                raw = Draw.DrawMorganEnvs(
                    [(mol_ex, aix_ex, r_ex)],
                    molsPerRow=1, subImgSize=(210, 125),
                    useSVG=False,
                )
                png_bytes = raw if isinstance(raw, bytes) else raw.data
                img = PILImage.open(io.BytesIO(png_bytes))
                ax_img.imshow(np.array(img))
            except Exception:
                ax_img.text(0.5, 0.5, str(bid), ha="center", va="center",
                            fontsize=10, transform=ax_img.transAxes)
        else:
            ax_img.text(0.5, 0.5, "—", ha="center", va="center",
                        fontsize=10, transform=ax_img.transAxes)

        ax_img.set_xticks([])
        ax_img.set_yticks([])
        for sp in ax_img.spines.values():
            sp.set_visible(False)

        n_occ = int((df[feat] > 0).sum())
        ax_img.text(
            0.5, -0.08,
            f"Δmed={delta:+.3f} eV   |r|={abs(r_val):.3f}\n  n={n_present:,}",
            ha="center", va="top", fontsize=10,
            transform=ax_img.transAxes
        )

plt.savefig("pi2stage_delta_boxplots.png", dpi=400,
            bbox_inches="tight", facecolor="white")
plt.show()
print("Figure -> pi2stage_delta_boxplots.png")

In [ ]:
# ============================================================================
#  Outliers de la régression — molécules les plus mal prédites (jeu de TEST)
#  Pour chaque cible : les N molécules avec le plus grand résidu |prédit − réel|
#  Affichage : structure 2D  +  valeur réelle / prédite / erreur
# ============================================================================
from rdkit.Chem import Draw
from IPython.display import display as ipy_display

N_OUT = 5   # molécules affichées par extrême (sur-estimées / sous-estimées)

# Reconstruire le DataFrame du jeu de test (index dans df_reg)
_, idx_test = train_test_split(np.arange(len(df_reg)), test_size=0.2, random_state=42)
df_test = df_reg.iloc[idx_test].reset_index(drop=True)

for target in TARGETS:
    r        = results[target]
    residus  = r["y_pred_te"] - r["y_te"]          # prédit − réel
    df_test  = df_test.copy()
    df_test[f"réel_{target}"]   = r["y_te"]
    df_test[f"prédit_{target}"] = r["y_pred_te"]
    df_test[f"résidu_{target}"] = residus
    df_test[f"|résidu|_{target}"] = np.abs(residus)

    rmse = r["rmse_te"]
    print(f"{'━'*62}")
    print(f"  Outliers régression — {target}  "
          f"(RMSE test = {rmse:.3f} eV, n_test = {len(df_test):,})")
    print(f"{'━'*62}")

    for label, sub in [
        (f"⬇  {N_OUT} plus SOUS-estimées  (prédit << réel)",
         df_test.nsmallest(N_OUT, f"résidu_{target}")),
        (f"⬆  {N_OUT} plus SUR-estimées   (prédit >> réel)",
         df_test.nlargest(N_OUT, f"résidu_{target}")),
    ]:
        print(f"\n{label}")
        mols    = [Chem.MolFromSmiles(s) for s in sub["SMILES"]]
        legends = [
            f"réel={row[f'réel_{target}']:.2f}\nprédit={row[f'prédit_{target}']:.2f}\nerr={row[f'résidu_{target}']:+.2f} eV"
            for _, row in sub.iterrows()
        ]
        img = Draw.MolsToGridImage(
            mols,
            molsPerRow=N_OUT,
            subImgSize=(300, 230),
            legends=legends,
        )
        ipy_display(img)

        recap = sub[["SMILES",
                     f"réel_{target}",
                     f"prédit_{target}",
                     f"résidu_{target}"]].copy()
        recap.columns = ["SMILES", f"{target} réel", f"{target} prédit", "résidu (eV)"]
        recap.index = range(1, N_OUT + 1)
        try:
            display(recap)
        except NameError:
            print(recap.to_string())
    print()